In [3]:
class Signal:
    def __init__(self, value=0):
        self.value = value

class Channel:
    def __init__(self):
        self.valid = Signal(0)
        self.ready = Signal(0)
        self.data  = Signal(0)
        self.addr  = Signal(0)

class AXILiteSlave:
    def __init__(self):
        self.memory = {}

        # Channels
        self.aw = Channel()
        self.w  = Channel()
        self.b  = Channel()
        self.ar = Channel()
        self.r  = Channel()

    def step(self):
        # WRITE ADDRESS + DATA handshake
        if self.aw.valid.value and self.w.valid.value:
            addr = self.aw.addr.value
            data = self.w.data.value

            self.memory[addr] = data

            # Accept transaction
            self.aw.ready.value = 1
            self.w.ready.value = 1

            # Send response
            self.b.valid.value = 1
            self.b.data.value = 0  # OKAY response

        else:
            self.aw.ready.value = 0
            self.w.ready.value = 0

        # RESPONSE accepted
        if self.b.valid.value and self.b.ready.value:
            self.b.valid.value = 0

        # READ ADDRESS handshake
        if self.ar.valid.value:
            addr = self.ar.addr.value
            data = self.memory.get(addr, 0)

            self.ar.ready.value = 1

            self.r.valid.value = 1
            self.r.data.value = data

        else:
            self.ar.ready.value = 0

        # READ data accepted
        if self.r.valid.value and self.r.ready.value:
            self.r.valid.value = 0

class AXILiteMaster:
    def __init__(self, slave):
        self.slave = slave

    def write(self, addr, data):
        # Set address + data
        self.slave.aw.addr.value = addr
        self.slave.aw.valid.value = 1

        self.slave.w.data.value = data
        self.slave.w.valid.value = 1

        # Wait for ready
        while not (self.slave.aw.ready.value and self.slave.w.ready.value):
            self.slave.step()

        # Clear valid
        self.slave.aw.valid.value = 0
        self.slave.w.valid.value = 0

        # Wait for response
        self.slave.b.ready.value = 1
        while not self.slave.b.valid.value:
            self.slave.step()

        self.slave.b.ready.value = 0

    def read(self, addr):
        self.slave.ar.addr.value = addr
        self.slave.ar.valid.value = 1

        while not self.slave.ar.ready.value:
            self.slave.step()

        self.slave.ar.valid.value = 0

        self.slave.r.ready.value = 1
        while not self.slave.r.valid.value:
            self.slave.step()

        data = self.slave.r.data.value
        self.slave.r.ready.value = 0

        return data


slave = AXILiteSlave()
master = AXILiteMaster(slave)

# Write
master.write(0x10, 123)

# Read
val = master.read(0x10)

print("Read Value:", val)

Read Value: 123


In [4]:
print("myname is pranav ")


myname is pranav 


In [ ]:


def full_adder_bit(a , b , cin ):
    sum_bit = a^b^cin
    cout =  (a & b) | (b & cin) | (a & cin)
    return sum_bit , cout 


full_adder_bit(1, 0 , 0 )


(1, 0)

the ripple carry adder 


In [7]:
# this  is ripple carry adder


def ripple_carry_adder(a, b, bits=8):
    result = 0
    carry = 0
    
    for i in range(bits):
        ai = (a >> i) & 1
        bi = (b >> i) & 1
        
        sum_bit, carry = full_adder_bit(ai, bi, carry)
        result |= (sum_bit << i)
    
    return result, carry


res, c = ripple_carry_adder(5, 3, 4)
print("Sum:", res, "Carry:", c)

Sum: 8 Carry: 0


In [9]:
# this is for class 

class Adder:
    def __init__(self, bits=8):
      self.bits = bits

    def compute(self, a, b): 
        result = 0
        carry = 0 
        for i in range(self.bits):
             ai = (a >> i) & 1 
             bi = (b >> i) & 1
             sum_bit = ai ^ bi ^ carry
             carry = (ai & bi) | (bi & carry) | (ai & carry)
             result |= (sum_bit << i) 
        
        return result, carry
    

a1 = Adder()

a1.compute(1, 2)



(3, 0)

In [10]:
# this is for mux
class Mux:
    def __init__(self, bits=8):
        self.bits = bits

    def mux1(self, inputs, select):
        # inputs = list of 8 values
        if select < 0 or select >= len(inputs):
            raise ValueError("Invalid select")

        return inputs[select]
    


mux = Mux(bits=8)

inputs = [10, 20, 30, 40, 50, 60, 70, 80]

out = mux.mux1(inputs, 3)

print("Output:", out)  # 40




Output: 40


this is axi master and slave for TPU 